In [1]:
# Run from the repo root so relative paths (models/, splits/, results/) resolve
import os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)

# Quickstart: datasets, splits and models

This notebook checks that the environment works and shows how to use the `cxr` package:

1. load a dataset and the six development datasets,
2. link them to the published splits in `splits/`,
3. load a model from `models/` and run it.

**Before you start:** download the original datasets and set the `CXR_DATASETS_PATH` environment variable (the folder that contains `nih/`, `CheXpert-v1.0/`, `MIMIC-CXR-JPG/`, ...) **before launching Jupyter**, for example:

```bash
CXR_DATASETS_PATH=/data/ds-CXR uv run jupyter lab
```

You can also edit `DATASETS_PATH` in `cxr/config.py`. The first time a dataset is loaded it is read and processed from the original files; it can then be cached with `ds.save_processed_dataset()` to speed up later loads.

In [2]:
import logging

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from tensorflow.keras.preprocessing.image import img_to_array, load_img

from cxr import cxr, config
from cxr.logger import setup_logging

setup_logging()
pd.set_option("display.max_columns", 60)
print("Datasets folder:", config.DATASETS_PATH)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

Datasets folder: /path/to/ds-CXR
TensorFlow: 2.15.1
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. A single dataset

Each dataset is a class in `cxr.cxr` that exposes a unified `DataFrame` (`ds.df`): the same columns and labels (`pneumothorax`, `support_devices`, `no_finding`, ...) for every dataset.

In [3]:
ds = cxr.NIHDataset()
ds.load()
ds.df[["patient_id", "cxr_projection", "dataset", "pneumothorax", "support_devices", "no_finding", "filepath"]].head()

2026-10-21 01:23:09 - INFO - cxr.py:176 - Processed dataset  loaded from /path/to/ds-CXR/nih


,patient_id,cxr_projection,dataset,pneumothorax,support_devices,no_finding,filepath
0,1,PA,nih,0,NaN,0,/path/to/ds-CXR/nih/images_001/images...
1,1,PA,nih,0,NaN,0,/path/to/ds-CXR/nih/images_001/images...
2,1,PA,nih,0,NaN,0,/path/to/ds-CXR/nih/images_001/images...
3,2,PA,nih,0,NaN,1,/path/to/ds-CXR/nih/images_001/images...
4,3,PA,nih,0,NaN,0,/path/to/ds-CXR/nih/images_001/images...


In [4]:
ds.describe_dataset()

Dataset LABEL: nih
Dataset PATH: /path/to/ds-CXR/nih)
IMAGES from each df: 
dataset
nih    112120
Name: count, dtype: int64

Unique patients: 30805

Patients' age: 46.63 +- 16.6

Patients' sex:
patient_sex
M    63340
F    48780
Name: count, dtype: int64

Original data SPLIT: 
test_split
0    0.771709
1    0.228291
Name: proportion, dtype: float64



## 2. The six development datasets

`load_datasets(..., combine=True)` returns a single object with all the images. It should report **904,454** images before projection filtering and cleaning (see `docs/methodology.md`).

In [5]:
from cxr.config import PNEUMOTHORAX_TRAINING_DATASETS

dev = cxr.load_datasets(datasets_to_load=PNEUMOTHORAX_TRAINING_DATASETS, combine=True)
df = dev.df
summary = df.groupby("dataset").agg(images=("filepath", "size"), pneumothorax_rate=("pneumothorax", "mean"))
summary.loc["TOTAL"] = [len(df), df.pneumothorax.mean()]
summary.round(3)

2026-10-21 01:23:09 - WARNING - cxr.py:716 - For now, you need to include metadata, split and chexpert labels from MIMIC-CXR-JPG in the folder


2026-10-21 01:23:09 - INFO - cxr.py:81 - Loading dataset nih...


2026-10-21 01:23:10 - INFO - cxr.py:176 - Processed dataset  loaded from /path/to/ds-CXR/nih


2026-10-21 01:23:10 - INFO - cxr.py:81 - Loading dataset CheXpert...


2026-10-21 01:23:10 - INFO - cxr.py:176 - Processed dataset  loaded from /path/to/ds-CXR/CheXpert-v1.0


...

2026-10-21 01:23:24 - INFO - cxr.py:81 - Loading dataset VinDr-CXR...


2026-10-21 01:23:24 - INFO - cxr.py:176 - Processed dataset  loaded from /path/to/ds-CXR/VinDR-CXR/


2026-10-21 01:23:24 - INFO - cxr.py:81 - Loading dataset SIIM-ACR...


2026-10-21 01:23:24 - INFO - cxr.py:176 - Processed dataset  loaded from /path/to/ds-CXR/siim-acr-pneumothorax-segmentation/abishek-siim-png-images/input


,images,pneumothorax_rate
dataset,,
CheXpert,224316.0,0.204
MIMIC-CXR,377110.0,0.172
PadChest,160861.0,0.004
SIIM-ACR,12047.0,0.222
VinDR-CXR,18000.0,0.005
nih,112120.0,0.046
TOTAL,904454.0,0.082


## 3. Published splits

`splits/development_splits.csv.gz` only contains identifiers and the `train`/`valid` assignment (paths relative to `DATASETS_PATH`). It is joined to the `DataFrame` by relative path.

In [6]:
def relative_path(series):
    """Path relative to the datasets root, independent of where your copy lives."""
    return series.str.replace(r"^.*?ds-CXR/", "", regex=True)

splits = pd.read_csv("splits/development_splits.csv.gz")
df["rel"] = relative_path(df.filepath)
dev_df = df.merge(splits[["filepath", "split", "in_multisource_training_set"]], left_on="rel", right_on="filepath", suffixes=("", "_split"))
print(f"{len(dev_df):,} of {len(splits):,} split images found in your data")
dev_df.groupby("split").agg(images=("filepath", "size"), patients=("patient_uuid", "nunique"), pneumothorax_rate=("pneumothorax", "mean")).round(3)

687,864 of 687,864 split images found in your data


,images,patients,pneumothorax_rate
split,,,
train,660329,246985,0.088
valid,27535,10293,0.082


In [7]:
# No patient may appear in both train and valid
patients = dev_df.groupby("split").patient_uuid.agg(set)
print("Patients shared between train and valid:", len(patients["train"] & patients["valid"]))
print("Unique images used to train the multisource model:", int(dev_df.in_multisource_training_set.sum()))

Patients shared between train and valid: 0
Unique images used to train the multisource model: 322569


## 4. Public external test

Datasets never used for training. `splits/external_test.csv.gz` lists the images used after filtering projections and problematic files.

The private SMS clinical dataset used for the additional external validation is not distributed with this repository; its aggregated results are in `results/sms_external_validation/`.

In [8]:
from cxr.config import PNEUMOTHORAX_TESTING_DATASETS

ext = cxr.load_datasets(datasets_to_load=PNEUMOTHORAX_TESTING_DATASETS, combine=True).df
ext["rel"] = relative_path(ext.filepath)
ext_splits = pd.read_csv("splits/external_test.csv.gz")
ext_df = ext.merge(ext_splits[["filepath"]], left_on="rel", right_on="filepath", suffixes=("", "_split"))
print(f"{len(ext_df):,} of {len(ext_splits):,} images found")
ext_df.groupby("dataset").agg(images=("filepath", "size"), pneumothorax_rate=("pneumothorax", "mean")).round(3)

2026-10-21 01:23:30 - INFO - cxr.py:81 - Loading dataset BRAX...


2026-10-21 01:23:30 - INFO - cxr.py:176 - Processed dataset  loaded from /path/to/ds-CXR/BRAX/1.1.0/


2026-10-21 01:23:30 - INFO - cxr.py:81 - Loading dataset CANDID-PTX...


2026-10-21 01:23:30 - WARNING - cxr.py:180 - Processed dataset not found. You need to run load_dataset() and preprocess_dataset() and then save_processed_dataset() to create it.


2026-10-21 01:23:30 - INFO - cxr.py:160 - Loading dataset with suffix=''...


...

2026-10-21 01:23:30 - INFO - cxr.py:162 - Preprocessing dataset...


2026-10-21 01:23:31 - INFO - cxr.py:81 - Loading dataset CRADI...


2026-10-21 01:23:31 - INFO - cxr.py:176 - Processed dataset  loaded from /path/to/ds-CXR/CRADI/


53,426 of 53,426 images found


,images,pneumothorax_rate
dataset,,
BRAX,19429,0.778
CANDID-PTX,19237,0.834
CRADI,10440,0.019
Indiana,3822,0.007
PTX498,498,1.000


## 5. Load a model and predict

The models in `models/` (see `MODELS.md`) are loaded whole with `load_model(..., compile=False)`. The input is an RGB image of 456×456 pixels with values in 0–255 (rescaling is part of the model) and the output is three probabilities: `pneumothorax`, `support_devices`, `no_finding`.

Quick check with 64 **validation** images from NIH (32 with pneumothorax and 32 without). This is only a smoke test, not an evaluation: for real metrics use `notebooks/evaluate/`.

In [9]:
targets = ["pneumothorax", "support_devices", "no_finding"]
model = tf.keras.models.load_model("models/effb5_ptx_multisource.keras", compile=False)
print(model.input_shape, "->", model.output_shape)

(None, 456, 456, 3) -> (None, 3)


In [10]:
sample = dev_df[(dev_df.split == "valid") & (dev_df.dataset == "nih")]
sample = pd.concat([
    sample[sample.pneumothorax == 1].sample(32, random_state=0),
    sample[sample.pneumothorax == 0].sample(32, random_state=0),
])

X = np.stack([img_to_array(load_img(p, target_size=(456, 456), color_mode="rgb")) for p in sample.filepath])
probs = pd.DataFrame(model.predict(X, batch_size=8, verbose=0), columns=targets, index=sample.index)

print("Pneumothorax AUC (64 NIH validation images):", round(roc_auc_score(sample.pneumothorax, probs.pneumothorax), 3))
probs.assign(true_pneumothorax=sample.pneumothorax).groupby("true_pneumothorax").pneumothorax.describe().round(3)

Pneumothorax AUC (64 NIH validation images): 0.979


,count,mean,std,min,25%,50%,75%,max
true_pneumothorax,,,,,,,,
0.0,32.0,0.032,0.060,0.000,0.001,0.010,0.025,0.283
1.0,32.0,0.636,0.305,0.026,0.416,0.721,0.870,0.996


## Next steps

- `notebooks/evaluate/`: full evaluation (validation, public external test and SMS external validation) of the 7 models.
- `notebooks/train/`: how the models were trained (documentation).
- `results/`: precomputed metrics and figures.
- `docs/methodology.md`: detailed training procedure.